# Model Evaluation Metrics, Overfitting & Underfitting

This notebook covers the evaluation of Machine Learning models, handling overfitting and underfitting, and understanding various metrics for both Regression and Classification problems.

## 1. Regression Evaluation Metrics

Regression metrics are used to measure the distance between the predicted values and the actual values.

### Common Metrics:
- **Mean Absolute Error (MAE):** Average of absolute differences between actual and predicted values. It's robust to outliers.
- **Mean Squared Error (MSE):** Average of squared differences. It penalizes large errors more heavily.
- **Root Mean Squared Error (RMSE):** Square root of MSE. It's in the same units as the target variable.
- **R-Squared ($R^2$):** Represents the proportion of variance for a dependent variable that's explained by an independent variable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Generate synthetic regression data
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate_regressor(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    print(f"--- {name} Metrics ---")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R-Squared: {r2:.4f}\n")

# 1. Linear Regression
evaluate_regressor("Linear Regression", LinearRegression(), X_train, X_test, y_train, y_test)

# 2. Decision Tree Regressor (Alternative Algorithm)
evaluate_regressor("Decision Tree Regressor", DecisionTreeRegressor(random_state=42), X_train, X_test, y_train, y_test)

## 2. Classification Evaluation Metrics

Classification metrics help us understand how well the model is separating different classes.

### Common Metrics:
- **Accuracy:** Fraction of correct predictions.
- **Precision:** Accuracy of positive predictions. $TP / (TP + FP)$
- **Recall (Sensitivity):** Fraction of positive instances correctly identified. $TP / (TP + FN)$
- **F1-Score:** Harmonic mean of Precision and Recall.
- **Confusion Matrix:** A table describing the performance of a classification model.
- **ROC-AUC:** Area under the Receiver Operating Characteristic curve. It measures the ability of a classifier to distinguish between classes.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# Generate synthetic classification data
X_c, y_c = make_classification(n_samples=1000, n_features=10, n_classes=2, random_state=42)

# Split the data
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

def evaluate_classifier(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} Metrics ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}\n")
    
    # Confusion Matrix Visualization for the last model evaluated
    return confusion_matrix(y_test, y_pred)

# 1. Logistic Regression
evaluate_classifier("Logistic Regression", LogisticRegression(), X_train_c, X_test_c, y_train_c, y_test_c)

# 2. Random Forest Classifier (Alternative Algorithm)
cm = evaluate_classifier("Random Forest Classifier", RandomForestClassifier(random_state=42), X_train_c, X_test_c, y_train_c, y_test_c)

# Confusion Matrix Visualization (Random Forest)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix (Random Forest)')
plt.show()

## 3. Bias, Variance, Underfitting & Overfitting

### Model Bias & Variance
- **Bias:** Error due to overly simplistic assumptions in the learning algorithm. High bias can cause the model to miss relevant relations between features and target outputs (**Underfitting**).
- **Variance:** Error due to too much complexity in the learning algorithm. High variance can cause the algorithm to model the random noise in the training data (**Overfitting**).

### Underfitting vs. Overfitting
1. **Underfitting:** Occurs when a model is too simple to learn the underlying structure of the data. It performs poorly on both training and test data.
2. **Overfitting:** Occurs when a model learns the training data too well, including its noise. It performs very well on training data but poorly on test data.
3. **Good Fit:** The goal is to find a balance where the model performs well on both.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

def plot_overfitting_underfitting():
    np.random.seed(0)
    n_samples = 30
    degrees = [1, 4, 15] # 1: Underfitting, 4: Good Fit, 15: Overfitting

    X = np.sort(np.random.rand(n_samples))
    y = np.cos(1.5 * np.pi * X) + np.random.randn(n_samples) * 0.1

    plt.figure(figsize=(14, 5))
    for i in range(len(degrees)):
        ax = plt.subplot(1, len(degrees), i + 1)
        plt.setp(ax, xticks=(), yticks=())

        polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
        linear_regression = LinearRegression()
        pipeline = Pipeline([("polynomial_features", polynomial_features),
                             ("linear_regression", linear_regression)])
        pipeline.fit(X[:, np.newaxis], y)

        X_test = np.linspace(0, 1, 100)
        plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
        plt.plot(X_test, np.cos(1.5 * np.pi * X_test), label="True function")
        plt.scatter(X, y, edgecolor='b', s=20, label="Samples")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.xlim((0, 1))
        plt.ylim((-2, 2))
        plt.legend(loc="best")
        plt.title(f"Degree {degrees[i]}\n(Bias vs Variance)")
    plt.show()

plot_overfitting_underfitting()